In [1]:
#Import Libraries
import pandas as pd
import numpy as np
import sqlite3

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

Load Cleaned Data From Database

In [2]:
conn = sqlite3.connect("creditpathai.db")
df = pd.read_sql("SELECT * FROM loan_data_cleaned", conn)
conn.close()

df.head()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,person_home_ownership_OTHER,...,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,loan_grade_B,loan_grade_C,loan_grade_D,loan_grade_E,loan_grade_F,loan_grade_G
0,22.0,59000.0,123.0,35000.0,16.02,1.0,0.59,1,3.0,0,...,0,0,1,0,0,0,1,0,0,0
1,21.0,9600.0,5.0,1000.0,11.14,0.0,0.10,0,2.0,0,...,0,0,0,0,1,0,0,0,0,0
2,25.0,9600.0,1.0,5500.0,12.87,1.0,0.57,0,3.0,0,...,0,1,0,0,0,1,0,0,0,0
3,23.0,65500.0,4.0,35000.0,15.23,1.0,0.53,0,2.0,0,...,0,1,0,0,0,1,0,0,0,0
4,24.0,54400.0,8.0,35000.0,14.27,1.0,0.55,1,4.0,0,...,0,1,0,0,0,1,0,0,0,0


#Define Features & Target

In [3]:
X = df.drop("loan_status", axis=1)
y = df["loan_status"]

#Train-Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

#MODEL 1 — XGBOOST
Train XGBoost

In [5]:
xgb_model = XGBClassifier(
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

Predictions

In [6]:
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:,1]

Calculate KPIs (XGBoost)

In [7]:
accuracy = accuracy_score(y_test, y_pred_xgb) * 100
precision = precision_score(y_test, y_pred_xgb) * 100
recall = recall_score(y_test, y_pred_xgb) * 100
f1 = f1_score(y_test, y_pred_xgb) * 100
auc = roc_auc_score(y_test, y_prob_xgb) * 100

print("\n" + "="*45)
print("        XGBOOST PERFORMANCE")
print("="*45)

print(f" Accuracy   : {accuracy:.2f}%")
print(f" Precision  : {precision:.2f}%")
print(f" Recall     : {recall:.2f}%")
print(f" F1 Score   : {f1:.2f}%")
print(f" ROC-AUC    : {auc:.2f}%")

print("="*45)


        XGBOOST PERFORMANCE
 Accuracy   : 93.51%
 Precision  : 95.11%
 Recall     : 74.14%
 F1 Score   : 83.33%
 ROC-AUC    : 94.65%


Train LightGBM

In [8]:
lgb_model = LGBMClassifier(random_state=42)

lgb_model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 4962, number of negative: 17729
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005572 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 981
[LightGBM] [Info] Number of data points in the train set: 22691, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.218677 -> initscore=-1.273393
[LightGBM] [Info] Start training from score -1.273393


LGBMClassifier(random_state=42)

Predictions

In [9]:
y_pred_lgb = lgb_model.predict(X_test)
y_prob_lgb = lgb_model.predict_proba(X_test)[:,1]

Calculate KPIs (LightGBM)


In [10]:
accuracy = accuracy_score(y_test, y_pred_lgb) * 100
precision = precision_score(y_test, y_pred_lgb) * 100
recall = recall_score(y_test, y_pred_lgb) * 100
f1 = f1_score(y_test, y_pred_lgb) * 100
auc = roc_auc_score(y_test, y_prob_lgb) * 100

print("\n" + "="*45)
print("        LIGHTGBM PERFORMANCE")
print("="*45)

print(f" Accuracy   : {accuracy:.2f}%")
print(f" Precision  : {precision:.2f}%")
print(f" Recall     : {recall:.2f}%")
print(f" F1 Score   : {f1:.2f}%")
print(f" ROC-AUC    : {auc:.2f}%")

print("="*45)


        LIGHTGBM PERFORMANCE
 Accuracy   : 93.67%
 Precision  : 97.25%
 Recall     : 73.11%
 F1 Score   : 83.47%
 ROC-AUC    : 94.69%
